In [1]:
import pickle
import pandas as pd
import numpy as np

import warnings
warnings.filterwarnings("ignore")

In [2]:
x_train = pickle.load(open("../models/x_train_reg.pkl","rb"))
x_test = pickle.load(open("../models/x_test_reg.pkl","rb"))

y_train = pickle.load(open("../models/y_train_reg.pkl","rb"))
y_test = pickle.load(open("../models/y_test_reg.pkl","rb"))

In [4]:
from sklearn.linear_model import (
    LinearRegression,
    Ridge,
    Lasso,
    ElasticNet
)

from sklearn.tree import DecisionTreeRegressor

from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor,
    ExtraTreesRegressor,
    AdaBoostRegressor
)

In [ ]:
# Define Models

models = {
    "Linear Regression": LinearRegression(),
    "Ridge": Ridge(),
    "Lasso": Lasso(),
    "ElasticNet": ElasticNet(),
    "Decision Tree": DecisionTreeRegressor(
        random_state=42
    ),
    "Random Forest": RandomForestRegressor(
        random_state=42
    ),
    "Gradient Boosting": GradientBoostingRegressor(
        random_state=42
    ),
    "Extra Trees": ExtraTreesRegressor(
        random_state=42
    ),
    "AdaBoost": AdaBoostRegressor(
        random_state=42
    )
}

In [7]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

In [6]:
# Train Models

results = []
best_model = None
best_r2 = -100

In [8]:
# Training Loop

for name, model in models.items():
    print(f"Training {name}")
    model.fit(x_train, y_train)
    prediction = model.predict(x_test)
    mae = mean_absolute_error(y_test, prediction)
    rmse = np.sqrt(mean_squared_error(y_test, prediction))
    r2 = r2_score(y_test, prediction)
    results.append([name, mae, rmse, r2])
    
    if r2 > best_r2:
        best_r2 = r2
        best_model = model

Training Linear Regression
Training Ridge
Training Lasso
Training ElasticNet
Training Decision Tree
Training Random Forest
Training Gradient Boosting
Training Extra Trees
Training AdaBoost


In [9]:
# Compare Models

results_df = pd.DataFrame(
    results,
    columns = ["Model", "MAE", "RMSE", "R2 Score"]
)

results_df.sort_values(
    by="R2 Score",
    ascending=False,
    inplace=True
)

results_df

,Model,MAE,RMSE,R2 Score
1,Ridge,0.798136,0.999014,0.606033
0,Linear Regression,0.798136,0.999014,0.606033
6,Gradient Boosting,0.801259,1.003166,0.602752
5,Random Forest,0.820276,1.026307,0.584213
7,Extra Trees,0.823224,1.030640,0.580695
8,AdaBoost,0.833504,1.044813,0.569083
3,ElasticNet,1.119441,1.402158,0.223914
4,Decision Tree,1.164165,1.458611,0.160163
2,Lasso,1.269383,1.591780,-0.000190


In [10]:
# Best Model

print(best_model)
print("Best R2 :", best_r2)

Ridge()
Best R2 : 0.6060332680926397


In [11]:
results_df.to_csv("../models/regression_results.csv", index=False)

****Hyperparameter Tuning of Ridge (GridSearchCV)****

In [12]:
from sklearn.model_selection import GridSearchCV

In [14]:
ridge_params = {
    "alpha":[0.001,0.01,0.1,1,10,100],
    "solver":[
        "auto",
        "svd",
        "cholesky",
        "lsqr",
        "sag"
    ]
}

ridge_search = GridSearchCV(
    estimator=Ridge(),
    param_grid=ridge_params,
    cv=5,
    scoring="r2",
    n_jobs=-1,
    verbose=2
)

ridge_search.fit(x_train, y_train)

best_ridge = ridge_search.best_estimator_

print(ridge_search.best_params_)

Fitting 5 folds for each of 30 candidates, totalling 150 fits
{'alpha': 10, 'solver': 'auto'}


In [16]:
ridge_pred = best_ridge.predict(x_test)

ridge_r2 = r2_score(y_test, ridge_pred)

ridge_mae = mean_absolute_error(y_test, ridge_pred)

ridge_rmse = np.sqrt(mean_squared_error(y_test, ridge_pred))

****Gradient Boosting (RandomizedSearchCV)****

In [17]:
from sklearn.model_selection import RandomizedSearchCV

In [18]:
gb_params = {

    "n_estimators":[100,200,300],

    "learning_rate":[0.01,0.05,0.1],

    "max_depth":[3,5,7],

    "subsample":[0.8,0.9,1.0],

    "min_samples_split":[2,5,10]
}

gb_search = RandomizedSearchCV(

    estimator=GradientBoostingRegressor(random_state=42),

    param_distributions=gb_params,

    n_iter=20,

    cv=3,

    scoring="r2",

    random_state=42,

    n_jobs=-1,

    verbose=2
)

gb_search.fit(x_train, y_train)

best_gb = gb_search.best_estimator_

print(gb_search.best_params_)

Fitting 3 folds for each of 20 candidates, totalling 60 fits
{'subsample': 0.9, 'n_estimators': 200, 'min_samples_split': 5, 'max_depth': 3, 'learning_rate': 0.1}


In [20]:
gb_pred = best_gb.predict(x_test)

gb_r2 = r2_score(y_test, gb_pred)

gb_mae = mean_absolute_error(y_test, gb_pred)

gb_rmse = np.sqrt(mean_squared_error(y_test, gb_pred))

In [21]:
# Final Comparison Table

comparison = pd.DataFrame({

    "Model":[

        "Baseline Ridge",

        "Tuned Ridge",

        "Baseline Gradient Boosting",

        "Tuned Gradient Boosting"

    ],

    "R2":[

        0.606033,

        ridge_r2,

        0.602752,

        gb_r2

    ],

    "MAE":[

        0.798136,

        ridge_mae,

        0.801259,

        gb_mae

    ],

    "RMSE":[

        0.999014,

        ridge_rmse,

        1.003166,

        gb_rmse

    ]

})

comparison.sort_values(
    by="R2",
    ascending=False
)

,Model,R2,MAE,RMSE
1,Tuned Ridge,0.606034,0.798135,0.999014
0,Baseline Ridge,0.606033,0.798136,0.999014
2,Baseline Gradient Boosting,0.602752,0.801259,1.003166
3,Tuned Gradient Boosting,0.602069,0.801724,1.004028


In [22]:
# Save the Best Model

if ridge_r2 >= gb_r2:
    final_package_model = best_ridge
    print("Selected Model : Ridge")

else:
    final_package_model = best_gb
    print("Selected Model : Gradient Boosting")

Selected Model : Ridge


In [23]:
import pickle

pickle.dump(
    final_package_model,
    open("../models/package_model.pkl","wb")
)

print("Package model saved successfully.")

Package model saved successfully.
